# Data Ingestion

### document datastructure 

In [28]:
"""
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source" : "test.txt",
        "pages": 1,
        "author": "Adhi"
    }
)
"""

'\ndoc = Document(\n    page_content="This is a test document.",\n    metadata={\n        "source" : "test.txt",\n        "pages": 1,\n        "author": "Adhi"\n    }\n)\n'

In [29]:
from langchain_core.documents import Document

In [30]:
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source" : "test.txt",
        "pages": 1,
        "author": "Adhi"
    }
)

doc

Document(metadata={'source': 'test.txt', 'pages': 1, 'author': 'Adhi'}, page_content='This is a test document.')

Metadata is often underused. In production, you enrich it with things like chunk_index, source_url, ingestion_timestamp, document_type. This enables metadata filtering at retrieval time — e.g., "only retrieve from PDFs ingested after Jan 2025."

In [31]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/sample1.txt")

In [32]:
document = loader.load()
document

[Document(metadata={'source': '../data/text_files/sample1.txt'}, page_content='This is a sample text file for RAG pipeline testing.\nIt contains random information about machine learning and AI.\n\nMachine learning (ML) is a field of artificial intelligence (AI) that enables computers to learn from data and improve their performance over time without being explicitly programmed. ML algorithms are used in a wide variety of applications, such as email filtering, speech recognition, computer vision, and recommendation systems.\n\nDeep learning, a subset of ML, uses neural networks with many layers to model complex patterns in data. These models have achieved state-of-the-art results in tasks like image classification, natural language processing, and game playing.\n\nKey concepts in ML include supervised learning, unsupervised learning, reinforcement learning, and transfer learning. Supervised learning involves training a model on labeled data, while unsupervised learning deals with findi

In [33]:
file_paths = ["../data/text_files/sample1.txt", "../data/text_files/sample2.txt", "../data/text_files/sample3.txt"]
documents = []

for path in file_paths:
    loader = TextLoader(path)
    documents.extend(loader.load())

In [34]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "../data",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

documents = loader.load()


100%|██████████| 12/12 [00:02<00:00,  5.84it/s]


TextLoader — single file, single Document per file.

DirectoryLoader — batch loading. The glob pattern controls which files are picked up. loader_cls is the actual parser.

Pro insights:

- `PyMuPDFLoader` > `PyPDFLoader` for quality — it preserves layout better and is faster
- `UnstructuredPDFLoader` is best for complex PDFs (tables, mixed layouts) but is heavier
- For production: use `show_progress=True` + async loading for large corpora
- Real pipelines often use blob stores `(S3, GCS)` with loaders like `S3FileLoader` or `AzureBlobStorageFileLoader`
- Lazy loading (`loader.lazy_load()`) is preferred over `.load()` for large files — it's a generator, avoiding memory spikes

In [35]:
documents[3].metadata

{'producer': 'pdfTeX-1.40.13',
 'creator': 'LaTeX with hyperref package',
 'creationdate': '2018-06-15T01:53:32-07:00',
 'source': '..\\data\\accurate_uncertanities_for_dl_using_calibrated_regression.pdf',
 'file_path': '..\\data\\accurate_uncertanities_for_dl_using_calibrated_regression.pdf',
 'total_pages': 9,
 'format': 'PDF 1.5',
 'title': 'Accurate Uncertainties for Deep Learning Using Calibrated Regression',
 'author': 'Volodymyr Kuleshov, Nathan Fenner, Stefano Ermon',
 'subject': 'Proceedings of the International Conference on Machine Learning 2018',
 'keywords': 'boring formatting information, machine learning, ICML',
 'moddate': '2018-06-15T01:53:32-07:00',
 'trapped': '',
 'modDate': "D:20180615015332-07'00'",
 'creationDate': "D:20180615015332-07'00'",
 'page': 3}

Here for different document types we jsut change loader_cls and the rest of the code remains same.
- For pdf : PyPDFLoader, PyMuPDFLoader, UnstructuredPDFLoader
- For text : TextLoader
- for csv : CSVLoader
- for word : UnstructuredWordDocumentLoader
- for ppt : UnstructuredPowerPointLoader
- for html : UnstructuredHTMLLoader
- for json : JSONLoader
- for xml : XMLLoader
- for audio : AudioTranscriptionLoader
- for video : VideoLoader

# Chunking

In [36]:
## replacing "-\n" with "" in the page content of the documents
for doc in documents:
    doc.page_content = doc.page_content.replace("-\n", "")
    doc.page_content = doc.page_content.replace("\n", " ")


documents[0].page_content

'Accurate Uncertainties for Deep Learning Using Calibrated Regression Volodymyr Kuleshov 1 2 Nathan Fenner 2 Stefano Ermon 1 Abstract Methods for reasoning under uncertainty are a key building block of accurate and reliable machine learning systems. Bayesian methods provide a general framework to quantify uncertainty. However, because of model misspeciﬁcation and the use of approximate inference, Bayesian uncertainty estimates are often inaccurate — for example, a 90% credible interval may not contain the true outcome 90% of the time. Here, we propose a simple procedure for calibrating any regression algorithm; when applied to Bayesian and probabilistic models, it is guaranteed to produce calibrated uncertainty estimates given enough data. Our procedure is inspired by Platt scaling and extends previous work on classiﬁcation. We evaluate this approach on Bayesian linear regression, feedforward, and recurrent neural networks, and ﬁnd that it consistently outputs well-calibrated credible 

- This is called pre-chunking normalization and is often skipped by beginners — it matters a lot for retrieval quality
- More aggressive pipelines use regex to remove headers/footers, page numbers, repeated boilerplate
- Some teams use ftfy library to fix text encoding issues
- For scanned PDFs, you'd add an OCR step (Tesseract / AWS Textract / Azure Document Intelligence) before this

In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter   

*Most tuned commponent in RAG*

In [38]:
def split_documents(doucments, chunk_size= 1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""])
    return text_splitter.split_documents(doucments)


# seprator dictate the semantic boundaries and the quality of the chunks. The more natural the separator, the better the retrieval performance. For example, splitting on paragraphs ("\n\n") is often better than splitting on spaces (" ") because it preserves more context in each chunk.

*Key params*:

- chunk_size=1000 — characters per chunk. Too small = loses context. Too large = dilutes relevance signal.
- chunk_overlap=200 — overlap between adjacent chunks. Ensures context isn't cut off at boundaries. Usually 10–20% of chunk_size.
- length_function=len — you can swap this with a token counter (tiktoken) to respect LLM context windows precisely

| Splitting Strategy | Pros | Cons | Use Cases |
|--------------------|------|------|-----------|
| RecursiveCharacterTextSplitter | Simple, fast, good for clean text | Can produce uneven chunks, may split sentences | General-purpose, when you just need quick chunking |
| TokenTextSplitter | Respects token limits, better for LLM context windows | Requires tokenization step, slower | When you need precise control over chunk size in tokens (token budget) |
| MarkdownHeadingTextSplitter | Preserves document structure, good for markdown | Only works well with markdown, may produce very small chunks | When working with markdown documents and you want to preserve headings |
| HTMLHeadingTextSplitter | Preserves HTML structure, good for web content | Only works well with HTML, may produce very small chunks | When working with HTML documents and you want to preserve headings |
| SemanticChunker(Langchain) | Uses embeddings to create semantically meaningful chunks | More complex, requires LLM calls, slower | When you want high-quality chunks that preserve meaning, especially for complex documents |

In [39]:
splitted_docs = split_documents(documents)

splitted_docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.13', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-15T01:53:32-07:00', 'source': '..\\data\\accurate_uncertanities_for_dl_using_calibrated_regression.pdf', 'file_path': '..\\data\\accurate_uncertanities_for_dl_using_calibrated_regression.pdf', 'total_pages': 9, 'format': 'PDF 1.5', 'title': 'Accurate Uncertainties for Deep Learning Using Calibrated Regression', 'author': 'Volodymyr Kuleshov, Nathan Fenner, Stefano Ermon', 'subject': 'Proceedings of the International Conference on Machine Learning 2018', 'keywords': 'boring formatting information, machine learning, ICML', 'moddate': '2018-06-15T01:53:32-07:00', 'trapped': '', 'modDate': "D:20180615015332-07'00'", 'creationDate': "D:20180615015332-07'00'", 'page': 0}, page_content='Accurate Uncertainties for Deep Learning Using Calibrated Regression Volodymyr Kuleshov 1 2 Nathan Fenner 2 Stefano Ermon 1 Abstract Methods for reasoning under uncertainty are a key building blo

*SemanticChunker* is the production-grade approach: instead of fixed character count, it embeds sentences and splits where cosine similarity drops. Much better retrieval quality, higher compute cost.

*Chunk size* is a critical hyperparameter that you should tune based on your use case and document type:
- Q&A over dense technical docs → smaller chunks (256–512 tokens)
- Summarization / broad queries → larger chunks (1024–2048 tokens)
- Code → split by function/class, not characters

# Embeddings

In [40]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings 
import uuid 
import os
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

## Creating embeddings

`all-MiniLM-L6-v2` is a great general purpose lightweight embedding model.

In [41]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", persist_directory: str = "chroma_db"):

        self.model_name = model_name 
        self.model = None 
        self._load_model()

    def _load_model(self):
        try: 
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise e


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded.")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e


    def __repr__(self):
        return f"EmbeddingManager(model_name={self.model_name})"


# initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1973.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


EmbeddingManager(model_name=all-MiniLM-L6-v2)

`model.encode(texts, show_progress_bar=True)` 

takes a list of strings, passes through transformer in batches and returns an `np.ndarray` of shape (num_texts, embedding_dim).   Each row is a dense vector representing the semantic meaning of the corresponding text chunk.

| Model | Dims | Notes | 
|-------|------|-------|
| `all-MiniLM-L6-v2` | 384 | Fast, good for general use, small vector size |
| `all-mpnet-base-v2` | 768 | Better quality, larger vector size, more compute, slower |
| `BAAI/bge-large-en-v1.5` | 1024 | state-of-art open source, production-grade |
| `text-embedding-3-small` | 1536 | OpenAI's latest small embedding, good quality, low cost |
| `text-embedding-3-large` | 3072 | OpenAI's latest large embedding, best quality, higher cost |
| `voyage-3` | 1024 | Cohere's latest embedding, good quality, low cost |

**Where to evaluate model**: MTEB leaderboard (Massive Text Embedding Benchmark) is the gold standard for evaluating embedding models across 80+ tasks (retrieval, clustering, classification). Look for models that perform well on tasks similar to your use case (e.g., retrieval tasks if you're building a search engine).

## VectorStore (chromadb)

`chromadb` is goo for local dev vector store. for production we scale to `Milvus` or `Pinecone`. The API is similar across these, so you can start local and switch to cloud later with minimal code changes.

**Milvus** connection example:

```python
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType
from langchain.vectorstores import Milvus
# Define Milvus collection schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=768),
    FieldSchema(name="metadata", dtype=DataType.JSON)
]
schema = CollectionSchema(fields, description="RAG document embeddings")
# Create Milvus collection
collection = Collection(name="rag_embeddings", schema=schema)
# Connect Langchain VectorStore to Milvus
vectorstore = Milvus(collection=collection, embedding_function=embedding_model)
```

In [42]:
class VectorStore: 

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try: 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                                                                   metadata={"description": "Collection of PDF document embeddings"})
            
            print(f"Vector store initialized at {self.persist_directory} with collection '{self.collection_name}'")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e
        

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        try:

            if len(documents) != len(embeddings):
                raise ValueError("Number of documents and embeddings must match.")
            # it should match because we are generating embedding for each document and then adding it to the vector store

            ids = [] 
            metadatas = []
            documents_text = []
            embeddings_list = [] 

            for i , (doc, embedding) in enumerate(zip(documents, embeddings)):

                doc_id = f"doc_{uuid.uuid4().hex[:4]}_{i}"
                ids.append(doc_id)

                metadata = dict(doc.metadata) if doc.metadata else {}
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                documents_text.append(doc.page_content)

                embeddings_list.append(embedding.tolist())

                try: 
                    self.collection.add(
                        ids=[doc_id],
                        embeddings=[embedding.tolist()],
                        metadatas=[metadata],
                        documents=[doc.page_content]
                    )

                    # print(f"Added document {doc_id} to vector store.")

                except Exception as e:
                    print(f"Error adding document {doc_id} to vector store: {e}")
                    continue
            print(f"Added {len(ids)} documents to vector store.")

        except Exception as e:
            print(f"Error in add_documents: {e}")
            raise e
        

vectorstore = VectorStore()
vectorstore 

Vector store initialized at ../data/vector_store with collection 'pdf_documents'


HNSW (Hierarchical Navigable Small World) is the ANN index algorithm most vector DBs use. Key params interviewers love to ask:

- ef_construction — index build quality vs speed tradeoff
- M — number of connections per node, affects recall

In [43]:
## converting text to embeddings 
texts = [doc.page_content for doc in splitted_docs]

## generate embedding 
# embeddings = embedding_manager.generate_embeddings(texts)

## store in vector db 
# vectorstore.add_documents(splitted_docs, embeddings)

# Retrieval

In [44]:
class RAGRetriever: 
    def __init__(self, vectorstore : VectorStore, embedding_manager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager
        # self.top_k = top_k

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # converts user query into dense vector using same model used at index time. 

        try: 
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                # include=["metadatas", "documents", "embeddings"]
            )

            # runs Approximate Nearest Neighbor search via HNSW index. returns top_k most similar vectors. 
            # returns nested lists 

            retrieved_docs = []
            if results["documents"] and results["documents"][0]: 
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })      

                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

            else: 
                print("No docs")

            return retrieved_docs  

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise e


rag_ret = RAGRetriever(vectorstore, embedding_manager)

Here we have implemented **dense retrieval** ( semantic similairty of chunks using cosine similarity of embeddings). 

Production systems often use **Hybrid retrieval** — a combination of dense retrieval + sparse keyword matching (BM25). This can be implemented by first retrieving a candidate set with BM25, then re-ranking with embedding similarity.

Reranking — after retrieving top_k=20, pass results through a cross-encoder reranker (e.g., cross-encoder/ms-marco-MiniLM-L-6-v2) which reranks with full query-document attention. Return top 5 to the LLM. This is the biggest quality improvement after basic RAG.

MMR (Maximum Marginal Relevance) — LangChain retrievers support search_type="mmr". Instead of top-k by score, it balances relevance + diversity to avoid returning 5 nearly identical chunks.

Contextual compression — LangChain's ContextualCompressionRetriever wraps any retriever and uses an LLM to extract only the relevant sentences from each chunk before passing to generation. Reduces noise.

In [45]:
rag_ret.retrieve("Knowledge graph construction from text")

Batches: 100%|██████████| 1/1 [00:00<00:00, 25.36it/s]

Retrieved 5 documents for query: 'Knowledge graph construction from text' with top_k=5 and score_threshold=0.0


[{'id': 'doc_349a_144',
  'content': 'Knowledge graphs (KGs) [26] integrate collections of real-world entities connected by semantically meaningful relations. They usually store structured factual knowledge that allows easy access and information retrieval. KGs have been used in a variety of applications including recommendation systems [23], question answering Figure 1: TKGCon: Given a set of theme-specific documents, automatic construction of a theme-specific knowledge graph. [45], intelligent conversational systems [33], and medical concept modeling [17]. Existing knowledge graphs can be categorized into generic open-world KGs including Wikidata1 and domain-specific KGs including UMLS [9]. Despite the broad applications of knowledge graphs, there are two major issues attached to the existing KGs, even in the current era of large language models (LLMs). The first issue is the limited information granularity of existing KGs. Existing KGs, including the domain-specific ones, often inte

In [46]:
rag_ret.retrieve("Dogs are good", top_k=3, score_threshold=0.1)

Batches: 100%|██████████| 1/1 [00:00<00:00, 43.76it/s]

Retrieved 0 documents for query: 'Dogs are good' with top_k=3 and score_threshold=0.1


[]

# RAG Pipeline 

In [47]:
from dotenv import load_dotenv 

load_dotenv() 


True

In [51]:
# print(os.getenv("GROQ_API_KEY"))

In [50]:
from langchain_groq import ChatGroq 
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [61]:
class GroqLLM: 

    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str = None): 

        self.model_name = model_name
        self.api_key = api_key or os.getenv("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("GROQ_API_KEY not found in environment variables.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key, 
            model_name = self.model_name, 
            temperature=0.1, 
            max_tokens = 1024
        )

    def generate_response(self, query: str, context:str , max_length: int = 500): 

        prompt_template = PromptTemplate(
            input_variables = ["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
                Context:
                {context}

                Question: {question}

                Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
            )
        
        formatted_prompt = prompt_template.format(context=context, question=query)

        try: 
            messages = [HumanMessage(content = formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content 
        

        except Exception as e:
            return f"Error generating response: {e}"
            # raise e


    def __repr__(self):
        return f"GroqLLM(model_name={self.model_name})"

        

In [62]:
groq = GroqLLM()

## Integration

In [64]:
query = "What are the main challenges in knowledge graph construction from text?"
context = "Knowledge graph construction from text involves several challenges, including entity recognition and disambiguation, relation extraction, handling of ambiguous language, scalability issues with large corpora, and ensuring the accuracy and completeness of the extracted information. Additionally, integrating information from diverse sources and maintaining the knowledge graph over time can also be complex tasks."

response = groq.generate_response(query=query, context=context)

print(response)

The main challenges in building a knowledge graph from text are:

1. **Entity recognition and disambiguation** – correctly spotting entities and linking them to the right real‑world concepts.  
2. **Relation extraction** – identifying and classifying the relationships that connect entities.  
3. **Ambiguous language** – dealing with polysemy, coreference, and context‑dependent meanings.  
4. **Scalability** – processing very large corpora efficiently while preserving quality.  
5. **Accuracy and completeness** – ensuring the extracted facts are correct and that the graph covers all relevant information.  
6. **Integration of diverse sources** – merging data from multiple documents or domains without conflicts.  
7. **Maintenance over time** – updating the graph as new information arrives and as existing knowledge changes.


## Implementing a piepline

In [65]:
import time 

In [ ]:
class RAGPipeline: 

    def __init__(self, retriever, llm): 
        self.retriever = retriever 
        self.llm = llm 
        self.history = [] # storing query history 

    def query(self, question: str, top_k : int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]: 
        results = self.retriever.retrieve(query=question, top_k=top_k, score_threshold=min_score)

        if not results: 
            answer = "No relevant documents found to answer the question."
            sources = [] 
            context = "" 

        else: 
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source', 'unknown'),
                'page' : doc['metadata'].get('page', 'unknown'),
                'similarity_score': doc['similarity_score'],
                'preview': doc['content'][:200] + '...'
            } for doc in results]

            prompt = f"""Use the following context to answer the question concisely. \n Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"""

            if stream: 
                print("Streaming response:")
                for i in range(0, len(prompt),80):
                    print(prompt[i:i+80], end="", flush=True)
                    time.sleep(0.1)
                print()

            answer = self.llm.generate_response(query=question, context=context)

            # answer = response.content 

        citations = "\n".join([f"{i+1}. {src['source']} (Page: {src['page']}, Similarity: {src['similarity_score']:.2f})" for i, src in enumerate(sources)])
        answer_with_citations = f"{answer}\n\nSources:\n{citations}" if citations else answer

        summary = None 
        if summarize and answer: 
            summary_prompt = f"""Summarize the following answer concisely:\n\n{answer}"""
            summary = self.llm.generate_response(query = summary_prompt, context="")

        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'answer': answer,
            'answer_with_citations': answer_with_citations,
            'sources': sources,
            'summary': summary
        }



In [82]:
rag = RAGPipeline(retriever=rag_ret, llm=groq)



In [84]:
response = rag.query("What are the main challenges in knowledge graph construction from text?", top_k=3, min_score=0.1, stream=True, summarize=True)


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.39it/s]

Retrieved 3 documents for query: 'What are the main challenges in knowledge graph construction from text?' with top_k=3 and score_threshold=0.1
Streaming response:
Use the following context to answer the question concisely. 
 Context:
Knowledge graphs (KGs) [26] integrate collections of real-world entities connected by sem

antically meaningful relations. They usually store structured factual knowledge that allows easy access and information retrieval. KGs have been used in a variety of applications including recommendation systems [23], question answering Figure 1: TKGCon: Given a set of theme-specific documents, automatic construction of a theme-specific knowledge graph. [45], intelligent conversational systems [33], and medical concept modeling [17]. Existing knowledge graphs can be categorized into generic open-world KGs including Wikidata1 and domain-specific KGs including UMLS [9]. Despite the broad applications of knowledge graphs, there are two major issues attached to the existing KGs, even in the current era of large language models (LLMs). The first issue is the limited information granularity of existing KGs. Existing KGs, including the domain-specific ones, often integrate numerous sources of texts and

Automated Construction of Theme-specific Knowledge Graphs Linyi Ding University of Illinoi

In [85]:
print(response['answer'])

The literature highlights two core obstacles when building knowledge graphs directly from text:

1. **Limited information granularity** – Existing KGs (both generic and domain‑specific) tend to merge many sources into coarse facts, making it hard to retrieve fine‑grained, context‑specific knowledge.

2. **Deficiency in timeliness** – KGs are often static snapshots. They struggle to capture rapidly evolving events (e.g., breaking news, disasters) or keep up‑to‑date with new domain knowledge.

These challenges hinder the ability to construct theme‑specific, up‑to‑date, and highly detailed knowledge graphs from textual data.


### Query transformation — before retrieval, rewrite or expand the query:

- HyDE (Hypothetical Document Embeddings): Generate a fake answer to the query, embed that, use it as the query vector. Often improves recall significantly.
- Multi-query retrieval: Generate 3-5 rephrased versions of the query, retrieve for each, deduplicate results.
- Step-back prompting: Abstract the query to a higher level before retrieval.

### Chunking strategies you should know:

- Parent-child chunking: Index small chunks for retrieval precision, but return their larger parent chunk to the LLM for context. LangChain's `ParentDocumentRetriever`.
- Proposition chunking: Use an LLM to decompose documents into atomic factual statements before indexing.

### Evaluation — critical in interviews:

- RAGAS — framework to evaluate RAG: `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`
- Without eval, you're flying blind. Every change to chunk size, embedding model, or retriever should be measured.


### Indexing freshness:

- Incremental indexing — only embed changed/new documents
- Deduplication — hash `page_content` to skip re-embedding unchanged chunks


### Guardrails:

- Hallucination detection: Check if answer is grounded in retrieved context
- `score_threshold` like you have is a basic form — if max similarity < threshold, respond "I don't know"